In [0]:
# Task: Load hr_broker domain data to bronze layer
# Author: Virendra Dilip Tambavekar

In [0]:
%run ../../02_common_utils/landing_to_bronze

In [0]:
%run ../../02_common_utils/operations

In [0]:
recon_df=load_domain_to_bronze("hr_broker", spark)

In [0]:
# Configuration
CATALOG = "charles_schwab_retailbrokerage_dev_team_lemma"
spark.sql(f"USE CATALOG {CATALOG}")

# Extract carry-forwarded run_id from Bronze layer (originated in raw_to_landing_hr)
carried_run_id = str(spark.table(f"{CATALOG}.bronze.hr").select("`_run_id`").first()[0])
source_count = recon_df.select("source_landing_count").first()[0]
target_count = recon_df.select("target_bronze_total").first()[0]
print(f"Carry-forwarded run_id: {carried_run_id}")

# __ start_pipeline_run -- imported from operations
start_pipeline_run(spark=spark, run_id=carried_run_id, batch="ALL")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="RUNNING")

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="landing_to_bronze_hr", message="Pipeline started: Landing to Bronze ingestion for HR domain")

# __ log_pipeline_recon -- imported from operations
log_pipeline_recon(
    spark=spark,
    run_id=carried_run_id,
    batch_id="ALL",
    domain="HR",
    table_name="hr_broker",
    source_layer="landing",
    target_layer="bronze",
    source_count=source_count,
    target_count=target_count
)

# __ log_audit_event -- imported from operations
log_audit_event(
    spark=spark,
    run_id=carried_run_id,
    batch="ALL",
    layer="bronze",
    table_name="hr_broker",
    operation="OVERWRITE",
    rows_affected=target_count
)

# __ log_pipeline_message -- imported from operations
log_pipeline_message(spark=spark, run_id=carried_run_id, log_level="INFO", module="landing_to_bronze_hr", message=f"Pipeline completed: {target_count} rows written to bronze.hr_broker")

# __ log_domain_run_status -- imported from operations
log_domain_run_status(spark=spark, run_id=carried_run_id, batch="ALL", domain_name="HR", status="COMPLETED")

# __ end_pipeline_run -- imported from operations
end_pipeline_run(spark=spark, run_id=carried_run_id, status="SUCCESS")

print(f"Operations logging complete for run_id: {carried_run_id}")

In [0]:
def log_dq_result(spark: SparkSession, run_id: str, table_name: str, rule_name: str, failed_rows: int, total_rows: int):
    """
    Logs the outcome of a Data Quality (DQ) check.
    """
    status = 'PASS' if failed_rows == 0 else 'FAIL'
    
    spark.sql(f"""
        INSERT INTO operations.dq_results 
        (run_id, table_name, rule_name, failed_rows, total_rows, dq_status)
        VALUES ('{run_id}', '{table_name}', '{rule_name}', {failed_rows}, {total_rows}, '{status}')
    """)